In [28]:
import pandas as pd
import sqlite3

df = pd.read_csv('../data/processed/insurance_cleaned.csv')

# Force recreate database
conn = sqlite3.connect('../data/processed/insurance.db')
df.to_sql('insurance_sales', conn, if_exists='replace', index=False)

# Verify columns in database
cursor = conn.cursor()
cursor.execute("PRAGMA table_info(insurance_sales)")
cols = [row[1] for row in cursor.fetchall()]
print("Columns in database table:")
print(cols)
print(f"\n✅ Database recreated with {len(df):,} records")

Columns in database table:
['AGENCY_ID', 'STAT_PROFILE_DATE_YEAR', 'POLY_INFORCE_QTY', 'NB_WRTN_PREM_AMT', 'PRD_INCRD_LOSSES_AMT', 'RETENTION_RATIO', 'LOSS_RATIO', 'GROWTH_RATE_3YR', 'ACTIVE_PRODUCERS', 'AGENCY_AGE', 'PREMIUM_GROWTH_YOY', 'PROD_LINE_ENC', 'STATE_ENC', 'MONTHS', 'LOG_WRTN_PREM_AMT', 'WRTN_PREM_AMT']

✅ Database recreated with 147,760 records


In [29]:
query1 = """
SELECT 
    STAT_PROFILE_DATE_YEAR AS Year,
    COUNT(*) AS Total_Records,
    ROUND(SUM(WRTN_PREM_AMT), 2) AS Total_Written_Premium,
    ROUND(AVG(WRTN_PREM_AMT), 2) AS Avg_Premium_Per_Record,
    ROUND(SUM(NB_WRTN_PREM_AMT), 2) AS Total_New_Business_Premium
FROM insurance_sales
GROUP BY STAT_PROFILE_DATE_YEAR
ORDER BY STAT_PROFILE_DATE_YEAR
"""

result1 = pd.read_sql_query(query1, conn)
print("Query 1: Annual Premium Summary")
print(result1.to_string(index=False))
result1.to_csv('../data/processed/sql_q1_annual_summary.csv', index=False)

Query 1: Annual Premium Summary
 Year  Total_Records  Total_Written_Premium  Avg_Premium_Per_Record  Total_New_Business_Premium
 2005          12693           274847855.33                21653.50                 33128543.64
 2006          14059           413410568.17                29405.40                 53804061.03
 2007          14329           409016576.34                28544.67                 44679899.65
 2008          14908           415823965.06                27892.67                 53784113.85
 2009          15273           413354701.58                27064.41                 50105748.60
 2010          15092           410811002.20                27220.45                 37370356.12
 2011          14623           402377564.49                27516.76                 27909844.77
 2012          15639           405967783.45                25958.68                 40334733.46
 2013          15576           427281524.28                27432.04                 58060760.09
 2014   

In [30]:
query2 = """
SELECT 
    AGENCY_ID,
    ROUND(SUM(WRTN_PREM_AMT), 2) AS Total_Premium,
    ROUND(AVG(RETENTION_RATIO), 4) AS Avg_Retention,
    ROUND(AVG(LOSS_RATIO), 4) AS Avg_Loss_Ratio,
    COUNT(*) AS Years_Active
FROM insurance_sales
GROUP BY AGENCY_ID
ORDER BY Total_Premium DESC
LIMIT 10
"""

result2 = pd.read_sql_query(query2, conn)
print("Query 2: Top 10 Agencies by Total Premium")
print(result2.to_string(index=False))
result2.to_csv('../data/processed/sql_q2_top_agencies.csv', index=False)

Query 2: Top 10 Agencies by Total Premium
 AGENCY_ID  Total_Premium  Avg_Retention  Avg_Loss_Ratio  Years_Active
      5468    43800420.74         0.8477          0.8449           276
      9733    32454406.91         0.8609          0.3010           412
      1786    31314651.21         0.8776          0.4401           449
       886    29763095.55         0.8879          0.4722           338
      8365    27711251.47         0.8807          0.2287           355
      9362    27167460.59         0.8694          0.0019           494
      4351    24867838.69         0.8616          0.2347           443
      8652    24638042.63         0.8576          0.5744           361
      9761    24410979.89         0.8567          0.2429           273
       562    23899354.22         0.8862          0.2695           320


In [31]:
query3 = """
SELECT
    STAT_PROFILE_DATE_YEAR AS Year,
    PROD_LINE_ENC,
    ROUND(SUM(WRTN_PREM_AMT), 2) AS Total_Premium,
    ROUND(AVG(GROWTH_RATE_3YR), 4) AS Avg_3Yr_Growth_Rate,
    ROUND(AVG(RETENTION_RATIO), 4) AS Avg_Retention
FROM insurance_sales
GROUP BY STAT_PROFILE_DATE_YEAR, PROD_LINE_ENC
ORDER BY PROD_LINE_ENC, STAT_PROFILE_DATE_YEAR
"""

result3 = pd.read_sql_query(query3, conn)
print(" Query 3: YoY Premium by Product Line")
print(result3.to_string(index=False))
result3.to_csv('../data/processed/sql_q3_product_line_yoy.csv', index=False)

 Query 3: YoY Premium by Product Line
 Year  PROD_LINE_ENC  Total_Premium  Avg_3Yr_Growth_Rate  Avg_Retention
 2005              0    95495332.21              -0.0156         0.8875
 2006              0   151997178.86              -0.0156         0.8875
 2007              0   152532681.59              -0.0156         0.8875
 2008              0   165522754.49               0.0432         0.8875
 2009              0   167900101.65               0.0413         0.8875
 2010              0   166666352.74               0.0290         0.8875
 2011              0   166544290.38               0.0291         0.8875
 2012              0   174505752.49               0.0236         0.8875
 2013              0   190573499.00               0.0342         0.8875
 2014              0   204441129.84               0.0545         0.8875
 2005              1   179352523.12              -0.0156         0.8325
 2006              1   261413389.31              -0.0156         0.8288
 2007              1   256

In [32]:
query4 = """
SELECT
    STATE_ENC,
    COUNT(DISTINCT AGENCY_ID) AS Unique_Agencies,
    ROUND(SUM(WRTN_PREM_AMT), 2) AS Total_Premium,
    ROUND(AVG(LOSS_RATIO), 4) AS Avg_Loss_Ratio,
    ROUND(AVG(RETENTION_RATIO), 4) AS Avg_Retention_Ratio,
    ROUND(SUM(NB_WRTN_PREM_AMT), 2) AS Total_New_Business
FROM insurance_sales
GROUP BY STATE_ENC
ORDER BY Total_Premium DESC
"""

result4 = pd.read_sql_query(query4, conn)
print("Query 4: State-wise Financial Health")
print(result4.to_string(index=False))
result4.to_csv('../data/processed/sql_q4_state_health.csv', index=False)

conn.close()

Query 4: State-wise Financial Health
 STATE_ENC  Unique_Agencies  Total_Premium  Avg_Loss_Ratio  Avg_Retention_Ratio  Total_New_Business
         3              832   2.351810e+09          0.5282               0.8682        219158272.40
         4              390   5.280010e+08          0.7641               0.8802         70329301.52
         1              363   4.690770e+08          0.6287               0.8680         74695836.54
         0              444   4.388524e+08          0.7583               0.8594         56222247.30
         5              185   1.824008e+08          0.8823               0.8761         21054516.55
         2              108   4.029395e+07          0.8632               0.8875         14182673.31
